In [1]:
#Importa las librerías necesarias
import requests
import pandas as pd
import numpy as np
from keys import *

#Define ciudad  y pais
city = "New York"
country = "US"

#Realiza la solicitud a la API de OpenWeatherMap con la clave y parámetros
response = requests.get(f'http://api.openweathermap.org/data/2.5/forecast/?q={city},{country}&appid={OWM_key}&units=metric&lang=en')

In [ ]:
# Convierte la respuesta de la API a formato JSON
data = response.json()

# Extrae la lista de pronósticos del JSON
forecast_list = data.get('list', [])

# Crea listas vacías para guardar los datos del clima
times = []
temperatures = []
humidities = []
weather_statuses = []
wind_speeds = []
rain_volumes = []
snow_volumes = []

#Recorre cada pronóstico y extrae los valores
for entry in forecast_list:
    times.append(entry.get('dt_txt', np.nan))
    temperatures.append(entry.get('main', {}).get('temp', np.nan))
    humidities.append(entry.get('main', {}).get('humidity', np.nan))
    weather_statuses.append(entry.get('weather', [{}])[0].get('main', np.nan))
    wind_speeds.append(entry.get('wind', {}).get('speed', np.nan))
    rain_volumes.append(entry.get('rain', {}).get('3h', np.nan))
    snow_volumes.append(entry.get('snow', {}).get('3h', np.nan))

# Crea un DataFrame con toda la información del clima
df = pd.DataFrame({
    'time': times,
    'temperature': temperatures,
    'humidity': humidities,
    'weather_status': weather_statuses,
    'wind_speed': wind_speeds,
    'rain_volume_3h': rain_volumes,
    'snow_volume_3h': snow_volumes
})

#Muestra las primeras filas del DataFrame
print(df.head())

      weather_datetime  temperature  humidity weather_status  wind  rain_qty  \
0  2025-11-03 09:00:00        10.91        72         Clouds  1.79       NaN   
1  2025-11-03 12:00:00        11.37        72         Clouds  1.55       NaN   
2  2025-11-03 15:00:00        15.82        62         Clouds  3.61       NaN   
3  2025-11-03 18:00:00        15.18        64         Clouds  4.09       NaN   
4  2025-11-03 21:00:00        14.77        61         Clouds  3.62       NaN   

   snow municipality_iso_country  
0   NaN                Berlin,DE  
1   NaN                Berlin,DE  
2   NaN                Berlin,DE  
3   NaN                Berlin,DE  
4   NaN                Berlin,DE  


In [13]:

import sqlalchemy
import pymysql

# ===============================
# INSERTAR NEW YORK EN city_pop (sin duplicados)
# ===============================

import pandas as pd
from sqlalchemy import create_engine, text

# 🔹 Conexión a tu base de datos MySQL
user = 'root'
password = 'july0905'
host = '127.0.0.1'
port = 3307
database = 'gans'

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")


In [14]:
# 🔹 Leer el archivo CSV
df = pd.read_csv('worldcities.csv')

# 🔹 Filtrar por la ciudad New York (ignorando mayúsculas/minúsculas)
ny = df[df['city_ascii'].str.lower() == 'new york'].copy()

# 🔹 Crear la columna combinada municipality_iso_country
ny['municipality_iso_country'] = ny['city_ascii'] + ',' + ny['iso2']

# 🔹 Seleccionar columnas para insertar
ny_to_insert = ny[['city_ascii', 'iso2', 'population', 'municipality_iso_country']].rename(
    columns={'city_ascii': 'city', 'iso2': 'country_code'}
)

# ===============================
# 1️⃣ ELIMINAR SI YA EXISTE
# ===============================
with engine.connect() as conn:
    delete_query = text("DELETE FROM city_pop WHERE municipality_iso_country = 'New York,US';")
    conn.execute(delete_query)
    conn.commit()

print("🗑️ Registro previo de 'New York,US' eliminado (si existía).")

# ===============================
# 2️⃣ INSERTAR DE NUEVO
# ===============================
ny_to_insert.to_sql('city_pop', con=engine, if_exists='append', index=False)
print("✅ New York insertado en city_pop correctamente.")

# ===============================
# 3️⃣ CONSULTAR PARA VERIFICAR
# ===============================
query = "SELECT * FROM city_pop WHERE municipality_iso_country = 'New York,US';"
result = pd.read_sql(query, engine)

display(result)


🗑️ Registro previo de 'New York,US' eliminado (si existía).
✅ New York insertado en city_pop correctamente.


,id,city,country_code,population,municipality_iso_country
0,3,New York,US,18713220,"New York,US"
